# ImCOF whole-range H reconstruction and H→Y transfer — v3

**Purpose:** resolve the missing verified `H_whole` for `ImCOF-TAEA`, using the **same `single` and full-range `tensor` q(RH,T) functions** as frozen v1.6.5. Use existing primary process Y (do not rerun the PADAC/TPD models).  
**Do not sum lower and upper P0 H.** Full-domain H also counts cross-branch consecutive transitions.  
**Gates:** exact input schema → canonical branch P0 regression → whole-domain supported-transition audit → primary Y schema and common-cycle mask check → pairwise transfer.  
**Interpretation:** H uses supported consecutive **hourly ambient states**; Y uses supported **cycles**, so matched *representation* and climate, **not identical temporal state support**. H is an equilibrium opportunity descriptor, not recovered CO₂ or productivity.  
**Source lineage:** `process_fidelity_analysis_drive_fixed_v1_6_5_climate_cache_key_fix.ipynb`, run `20261115_192400_v1.6.5-climate-cache-key-fix`.  
**Validation:** created locally and syntax-checked; actual run needs Colab Google Drive mount.

In [ ]:

from pathlib import Path
import os, pickle, itertools, json
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

# ---------- robust Colab Drive mount ----------
MYDRIVE = Path('/content/drive/MyDrive')

if not MYDRIVE.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        raise RuntimeError(
            "Google Drive is not mounted and Colab credential propagation failed. "
            "Use Runtime > Restart session (or reconnect the runtime), then run this cell again. "
            "If Drive is already visible in the Colab Files pane, do not remount it; "
            "the notebook will automatically reuse /content/drive/MyDrive."
        ) from e

assert MYDRIVE.exists(), (
    "Drive mount did not produce /content/drive/MyDrive. "
    "Reconnect/restart the Colab runtime and rerun this cell."
)

print('Drive ready:', MYDRIVE)

RUN = Path('/content/drive/MyDrive/process_fidelity_runs/20261115_192400_v1.6.5-climate-cache-key-fix')
STATE = RUN / '_state'

EQ_MASTER = Path('/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone/recent_experimental_results/manuscript_required_outputs_only/results/SI_machine_readable_supporting_data/MAIN_equilibrium_dataset_master.xlsx')
P0_OUT = Path('/content/drive/MyDrive/climate_screening_v1_3_1_uncertainty_audit/climate_screening_backbone/recent_experimental_results/manuscript_required_outputs_only/results/P0_submission_gate_outputs_2026-10-14/P0_tensor_outputs')
P0_BOUNDS = P0_OUT / 'P0_material_bounds.csv'

PATHS = {
    'forcing': STATE / 'forcing.parquet',
    'p0_regression': RUN / 'equilibrium_representation_regression.csv',
    'active_scores': RUN / 'ImCOF_active_thermal_climate_scores.csv',
    'env_scores': RUN / 'ImCOF_environmental_thermal_climate_scores.csv',
}
for k,p in PATHS.items():
    assert p.exists(), f'Missing {k}: {p}'
for p in [EQ_MASTER, P0_BOUNDS]:
    assert p.exists(), f'Missing canonical input: {p}'

OUT = RUN / 'ImCOF_wholeH_Y_transfer_verified_v3'
OUT.mkdir(parents=True, exist_ok=True)
print('Output:', OUT)
MATERIAL = 'ImCOF-TAEA'



## Schema gate and frozen interpolation reconstruction

Input paths and column names were inspected against existing v1.6.5 and Drive CSVs. The Excel sheet containing `material,T_C,RH_pct,q_mmol_g` is discovered by exact header match, as in the source adapter.

In [ ]:

# ---------- exact minimal P0 representation builder copied from v1.6.5 adapter logic ----------
req = {'material','T_C','RH_pct','q_mmol_g'}
xls = pd.ExcelFile(EQ_MASTER)
hit = None
for sh in xls.sheet_names:
    raw = pd.read_excel(EQ_MASTER, sheet_name=sh, header=None, nrows=20)
    for i in range(len(raw)):
        vals = {str(v).strip() for v in raw.iloc[i].tolist() if pd.notna(v)}
        if req <= vals:
            hit = (sh, i)
            break
    if hit:
        break
assert hit is not None, 'Canonical equilibrium sheet/header not found'

eq = pd.read_excel(EQ_MASTER, sheet_name=hit[0], header=hit[1]).dropna(axis=1,how='all').dropna(axis=0,how='all').copy()
eq.columns = [str(c).strip() for c in eq.columns]
for c in ['T_C','RH_pct','q_mmol_g']:
    eq[c] = pd.to_numeric(eq[c], errors='raise')

bounds = pd.read_csv(P0_BOUNDS)
for c in ['Tlow','Tref','Thigh','Rmin','Rmax']:
    bounds[c] = pd.to_numeric(bounds[c], errors='raise')

b = bounds[bounds.material.astype(str).eq(MATERIAL)]
assert len(b) == 1, f'Expected one ImCOF bounds row, found {len(b)}'
b = b.iloc[0]
d = eq[eq.material.astype(str).eq(MATERIAL)].copy()
assert len(d), 'ImCOF equilibrium rows missing'

def _slice_knots(T):
    z = d[np.isclose(d.T_C.to_numpy(float), float(T), atol=1e-9, rtol=0)].sort_values('RH_pct')
    assert len(z), f'Missing T slice {T}'
    assert not z.duplicated('RH_pct', keep=False).any(), f'Duplicate RH at T={T}'
    return z.RH_pct.to_numpy(float), z.q_mmol_g.to_numpy(float)

def _interp_rh(rh, x, y):
    rh = np.asarray(rh,float)
    out = np.full(rh.shape, np.nan, float)
    ok = np.isfinite(rh) & (rh >= x.min()) & (rh <= x.max())
    out[ok] = np.interp(rh[ok], x, y)
    return out

MEASURED_T_KNOTS = np.array(sorted(float(x) for x in d.T_C.unique() if b.Tlow <= float(x) <= b.Thigh), float)
slices = {T:_slice_knots(T) for T in MEASURED_T_KNOTS}
xr, yr = _slice_knots(b.Tref)

def q_single(T, RH):
    T = np.asarray(T,float); RH = np.asarray(RH,float)
    out = _interp_rh(RH, xr, yr)
    out[(T < b.Tlow) | (T > b.Thigh) | (RH < b.Rmin) | (RH > b.Rmax)] = np.nan
    return out

def q_tensor_branch(branch, T, RH):
    T = np.asarray(T,float); RH = np.asarray(RH,float)
    out = np.full(T.shape, np.nan, float)
    if branch == 'lower':
        lo, hi = float(b.Tlow), float(b.Tref)
    elif branch == 'upper':
        lo, hi = float(b.Tref), float(b.Thigh)
    else:
        raise ValueError(branch)
    tb = MEASURED_T_KNOTS[(MEASURED_T_KNOTS >= lo) & (MEASURED_T_KNOTS <= hi)]
    assert len(tb) >= 2 and np.isclose(tb[0],lo) and np.isclose(tb[-1],hi), (branch,tb,lo,hi)
    for j in range(len(tb)-1):
        t0,t1 = float(tb[j]),float(tb[j+1])
        x0,y0 = slices[t0]; x1,y1 = slices[t1]
        q0 = _interp_rh(RH,x0,y0); q1 = _interp_rh(RH,x1,y1)
        itm = (T >= t0) & ((T < t1) if j < len(tb)-2 else (T <= t1))
        ok = np.isfinite(q0) & np.isfinite(q1) & itm & (RH >= b.Rmin) & (RH <= b.Rmax)
        if ok.any():
            wgt = (T[ok]-t0)/(t1-t0)
            out[ok] = (1-wgt)*q0[ok] + wgt*q1[ok]
    return out

def q_tensor(T, RH):
    T=np.asarray(T,float); RH=np.asarray(RH,float)
    out=np.full(T.shape,np.nan,float)
    ml=np.isfinite(T)&(T<=b.Tref)
    mu=np.isfinite(T)&(T>=b.Tref)
    if ml.any(): out[ml]=q_tensor_branch('lower',T[ml],RH[ml])
    if mu.any(): out[mu]=q_tensor_branch('upper',T[mu],RH[mu])
    return out

print(dict(Tlow=float(b.Tlow), Tref=float(b.Tref), Thigh=float(b.Thigh),
           Rmin=float(b.Rmin), Rmax=float(b.Rmax), measured_T=MEASURED_T_KNOTS.tolist()))


In [ ]:
w = pd.read_parquet(PATHS['forcing']).copy()
if 'timestamp' not in w.columns and 'timestamp_utc' in w.columns:
    w['timestamp'] = w['timestamp_utc']
required_weather={'climate','timestamp','T_C','RH_pct'}
assert required_weather <= set(w.columns), sorted(required_weather-set(w.columns))
w['timestamp']=pd.to_datetime(w.timestamp, utc=True, errors='raise')
w['T_C']=pd.to_numeric(w['T_C'], errors='raise')
w['RH_pct']=pd.to_numeric(w['RH_pct'], errors='raise')
assert not w.duplicated(['climate','timestamp']).any(), 'Duplicate hourly records'
w=w.sort_values(['climate','timestamp']).reset_index(drop=True)
print('Weather columns:',list(w.columns),'rows:',len(w),'climates:',w.climate.nunique())
assert w.climate.nunique()==36, 'Unexpected climate panel — do not silently alter the original panel'


## Gate 1: exact P0 lower/upper H regression

This is an independent functional check on the exact interpolation and masking conventions; only then calculate whole-range H. Nonfinite reference/prediction presence must also match.

In [ ]:

p0 = pd.read_csv(PATHS['p0_regression'])
assert {'material','rep','key','reference','check'} <= set(p0.columns)
p0 = p0[(p0.material.astype(str)==MATERIAL) & p0['check'].astype(str).eq('P0_regression')].copy()
parts = p0['key'].astype(str).str.split('|',expand=True)
p0['branch']=parts[0]; p0['temporal_mode']=parts[1]; p0['climate']=parts[2]
p0=p0[p0.temporal_mode.eq('subsample_1h') & p0.rep.isin(['single','tensor'])].copy()
assert len(p0)>0 and not p0.duplicated(['rep','branch','climate']).any()

gate_rows=[]
for r in p0.itertuples(index=False):
    g=w[w.climate.astype(str).eq(str(r.climate))].sort_values('timestamp')
    T=g.T_C.to_numpy(float); RH=g.RH_pct.to_numpy(float)
    qs=q_single(T,RH)
    qt=q_tensor_branch(r.branch,T,RH)
    ok=np.isfinite(qs)&np.isfinite(qt)
    tr=ok[:-1]&ok[1:]
    q = qs if r.rep=='single' else qt
    H=float(0.5*np.abs(np.diff(q)[tr]).sum()) if tr.any() else np.nan
    gate_rows.append(dict(rep=r.rep,branch=r.branch,climate=r.climate,
                          H_local=H,H_saved=float(r.reference),
                          abs_err=abs(H-float(r.reference)) if np.isfinite(H) and np.isfinite(float(r.reference)) else np.nan))
G1=pd.DataFrame(gate_rows)
assert (np.isfinite(G1.H_local) == np.isfinite(G1.H_saved)).all(), 'P0 finite-mask regression failed'
finite=G1.dropna(subset=['abs_err'])
print(G1.groupby(['rep','branch']).agg(n=('climate','size'),max_abs_err=('abs_err','max')))
assert len(finite) and finite.abs_err.max() < 1e-8, f'P0 H regression failed: max abs err {finite.abs_err.max()}'
G1.to_csv(OUT/'ImCOF_P0_branch_regression_gate.csv',index=False)
print('GATE 1 PASS')


## Whole-range H: primary reconstruction

For a climate trajectory, **one global common hour-state mask** `isfinite(q_single) & isfinite(q_tensor)` is used for both representations; only adjacent weather records exactly 1 hour apart contribute. This protects against silently counting a jump across a missing weather record. The separate cross-branch contribution is recorded for audit (it is **not** an additional H component appended twice).  

The full q tensor is piecewise in temperature, preserving the original measured T/RH interpolation, including the mixed upper RH knot resolution; no extrapolation.

In [ ]:
rows=[]
for city,g in w.groupby('climate',sort=True):
    g=g.sort_values('timestamp').reset_index(drop=True)
    T=g.T_C.to_numpy(float); RH=g.RH_pct.to_numpy(float)
    ts=g.timestamp
    qs=q_single(T,RH); qt=q_tensor(T,RH)
    states=np.isfinite(qs)&np.isfinite(qt)
    adjacent=(ts.diff().dt.total_seconds().to_numpy()[1:]==3600)
    tr=states[:-1]&states[1:]&adjacent
    assert np.isfinite(qs[states]).all() and np.isfinite(qt[states]).all()
    cross=(T[:-1] < float(b.Tref)) & (T[1:] > float(b.Tref)) | ((T[:-1] > float(b.Tref)) & (T[1:] < float(b.Tref)))
    for rep,q in [('single',qs),('tensor',qt)]:
        delta=0.5*np.abs(np.diff(q))
        rows.append(dict(material=MATERIAL,climate=city,representation=rep,
           H_whole=float(delta[tr].sum()) if tr.any() else np.nan,
           H_cross_branch=float(delta[tr&cross].sum()) if (tr&cross).any() else 0.0,
           n_common_states=int(states.sum()),n_common_transitions=int(tr.sum()),
           n_cross_branch_transitions=int((tr&cross).sum()),
           n_weather_hours=int(len(g)),
           supported_hour_fraction=float(states.mean()),
           representation_domain='full_Tlow_to_Thigh',
           H_definition='half_total_variation_on_common_adjacent_hourly_transitions'))
WHOLE_H=pd.DataFrame(rows)
assert not WHOLE_H.duplicated(['climate','representation']).any()
assert WHOLE_H.groupby('climate').n_common_transitions.nunique().max()==1
assert WHOLE_H.groupby('climate').n_common_states.nunique().max()==1
assert (WHOLE_H.H_cross_branch.fillna(0)<=WHOLE_H.H_whole.fillna(0)+1e-10).all()
WHOLE_H.to_csv(OUT/'ImCOF_whole_range_H_verified.csv',index=False)
print('Whole-range H computed; per-climate support and cross-boundary audit saved')
display(WHOLE_H.groupby('representation').agg(n_climates=('climate','nunique'),n_with_H=('H_whole','count'),
    median_hour_fraction=('supported_hour_fraction','median'),median_transitions=('n_common_transitions','median'),
    total_crossing_transitions=('n_cross_branch_transitions','sum')))


## Gate 2: archived primary Y, schema and supported-cycle equality

Read the **existing** v1.6.5 primary Y; do not recompute or substitute endpoints. This gate is a consistency check of *stored cycle support and representation pairing*, not a full independent reconstruction of Y from TPD/thermal caches. Separate `active` / `environmental` outcomes are retained without equating them physically.

In [ ]:
a=pd.read_csv(PATHS['active_scores'])
e=pd.read_csv(PATHS['env_scores'])
need_a={'material','controller','window','ads_condition','climate','representation','Y_active','n_supported_cycles','supported_cycle_fraction'}
need_e={'material','controller','thermal_variant','theta','execution_tier','climate','representation','Y_env','n_supported_cycles','supported_cycle_fraction'}
assert need_a <= set(a.columns),sorted(need_a-set(a.columns))
assert need_e <= set(e.columns),sorted(need_e-set(e.columns))
a=a[(a.material==MATERIAL)&(a.controller=='active_thermal_80C')&
    (a.window=='primary')&(a.ads_condition=='end_of_window')].copy()
e=e[(e.material==MATERIAL)&(e.controller=='environmental_thermal_PADAC')&
    (e.thermal_variant=='primary')&(e.theta=='PADAC_13X_FIELD_REALISATION')&
    (e.execution_tier=='PRIMARY')].copy()
a=a.rename(columns={'Y_active':'Y'})
e=e.rename(columns={'Y_env':'Y'})
a['operating_mode']='active_80C'
e['operating_mode']='environmental_PADAC13X'
y=pd.concat([a,e],ignore_index=True)
assert len(y) and set(y.representation)=={'single','tensor'}
assert not y.duplicated(['operating_mode','climate','representation']).any()
cy=[]
for (mode,climate),g in y.groupby(['operating_mode','climate']):
    assert set(g.representation)=={'single','tensor'}, (mode,climate)
    s=g.set_index('representation')
    assert int(s.loc['single','n_supported_cycles'])==int(s.loc['tensor','n_supported_cycles']), (mode,climate,'cycle count')
    assert np.isclose(float(s.loc['single','supported_cycle_fraction']),float(s.loc['tensor','supported_cycle_fraction']),atol=1e-12), (mode,climate,'cycle fraction')
    cy.append(dict(operating_mode=mode,climate=climate,n_supported_cycles=int(s.loc['single','n_supported_cycles']),
        supported_cycle_fraction=float(s.loc['single','supported_cycle_fraction'])))
CYCLE_SUPPORT=pd.DataFrame(cy)
CYCLE_SUPPORT.to_csv(OUT/'ImCOF_primary_process_cycle_support_audit.csv',index=False)
Y_SCORES=y[['material','climate','representation','operating_mode','Y','n_supported_cycles','supported_cycle_fraction']].copy()
Y_SCORES.to_csv(OUT/'ImCOF_primary_process_Y_source_subset.csv',index=False)
print('Gate 2 PASS: primary Y schema, uniqueness and common cycle counts/fractions')
display(CYCLE_SUPPORT.groupby('operating_mode').agg(n=('climate','nunique'),
   median_supported_cycle_fraction=('supported_cycle_fraction','median'),
   min_supported_cycle_fraction=('supported_cycle_fraction','min')))


## Climate alignment and fidelity, with pairwise H→Y transfer

Primary transfer compares reversal **due to representation reduction** (`single` vs `tensor`) within each endpoint. A different statistic, H–Y **within a representation**, is recorded separately. Ties and finite masks are explicit. The result remains a **within-model diagnostic** because H uses hourly weather variation but Y uses sampled adsorption cycles.

In [ ]:
A=WHOLE_H[['climate','representation','H_whole','n_common_transitions','supported_hour_fraction']].merge(
    Y_SCORES[['climate','representation','operating_mode','Y','n_supported_cycles','supported_cycle_fraction']],
    on=['climate','representation'],how='inner',validate='one_to_many')
assert not A.duplicated(['operating_mode','climate','representation']).any()
A.to_csv(OUT/'ImCOF_wholeH_wholeY_aligned_climate_scores.csv',index=False)

def kt(x,y):
    from scipy.stats import kendalltau,spearmanr
    ix=np.isfinite(x)&np.isfinite(y)
    return (float(kendalltau(x[ix],y[ix],variant='b').statistic) if ix.sum()>=3 else np.nan,
            float(spearmanr(x[ix],y[ix]).statistic) if ix.sum()>=3 else np.nan)

pair_rows=[];summary=[];corr=[];elig=[]
for mode,g in A.groupby('operating_mode'):
    pH=g.pivot(index='climate',columns='representation',values='H_whole')
    pY=g.pivot(index='climate',columns='representation',values='Y')
    supp=g[g.representation=='tensor'].set_index('climate')[['n_common_transitions','supported_hour_fraction','n_supported_cycles','supported_cycle_fraction']]
    p=pH.rename(columns=lambda k:'H_'+k).join(pY.rename(columns=lambda k:'Y_'+k),how='inner').join(supp,how='inner')
    need=['H_single','H_tensor','Y_single','Y_tensor']
    p=p.replace([np.inf,-np.inf],np.nan).dropna(subset=need)
    assert len(p)>=3, (mode,len(p))
    clim=p.index.tolist()
    hs=p.H_single.to_numpy(float); ht=p.H_tensor.to_numpy(float)
    ys=p.Y_single.to_numpy(float); yt=p.Y_tensor.to_numpy(float)
    th,rh=kt(hs,ht); ty,ry=kt(ys,yt)
    for rep in ['single','tensor']:
        t,r=kt(p['H_'+rep].to_numpy(float),p['Y_'+rep].to_numpy(float))
        corr.append(dict(operating_mode=mode,representation=rep,n_climates=len(p),tau_H_vs_Y=t,rho_H_vs_Y=r))
    counts={'Hsafe_Ysafe':0,'Hsafe_Yrev':0,'Hrev_Ysafe':0,'Hrev_Yrev':0}
    n_htie=n_ytie=0
    for i in range(len(clim)):
        for j in range(i+1,len(clim)):
            dhs=hs[i]-hs[j]; dht=ht[i]-ht[j]; dys=ys[i]-ys[j]; dyt=yt[i]-yt[j]
            shs,sht,sys,syt=map(np.sign,(dhs,dht,dys,dyt))
            h_tie=shs==0 or sht==0; y_tie=sys==0 or syt==0
            h_rev=not h_tie and shs!=sht; y_rev=not y_tie and sys!=syt
            if h_tie:n_htie+=1
            if y_tie:n_ytie+=1
            status='EXCLUDED_TIE' if h_tie or y_tie else ('Hrev' if h_rev else 'Hsafe')+'_'+('Yrev' if y_rev else 'Ysafe')
            if status!='EXCLUDED_TIE':counts[status]+=1
            pair_rows.append(dict(operating_mode=mode,climate_i=clim[i],climate_j=clim[j],
                H_single_i=hs[i],H_single_j=hs[j],H_tensor_i=ht[i],H_tensor_j=ht[j],
                Y_single_i=ys[i],Y_single_j=ys[j],Y_tensor_i=yt[i],Y_tensor_j=yt[j],
                H_reversal=bool(h_rev),Y_reversal=bool(y_rev),H_tie=bool(h_tie),Y_tie=bool(y_tie),
                transfer_status=status,
                min_hour_transitions=int(min(p.iloc[i].n_common_transitions,p.iloc[j].n_common_transitions)),
                min_cycle_count=int(min(p.iloc[i].n_supported_cycles,p.iloc[j].n_supported_cycles)),
                min_cycle_fraction=float(min(p.iloc[i].supported_cycle_fraction,p.iloc[j].supported_cycle_fraction))))
    safe=counts['Hsafe_Ysafe']+counts['Hsafe_Yrev']; rev=counts['Hrev_Ysafe']+counts['Hrev_Yrev']; total=safe+rev
    summary.append(dict(material=MATERIAL,operating_mode=mode,n_climates=len(p),n_all_pairs=len(clim)*(len(clim)-1)//2,
       n_joint_strict_pairs=total,n_H_tie_pairs=n_htie,n_Y_tie_pairs=n_ytie,
       tau_H_single_tensor=th,tau_Y_single_tensor=ty,rho_H_single_tensor=rh,rho_Y_single_tensor=ry,
       **{'n_'+k:v for k,v in counts.items()},
       P_Yrev_given_Hsafe=counts['Hsafe_Yrev']/safe if safe else np.nan,
       P_Yrev_given_Hrev=counts['Hrev_Yrev']/rev if rev else np.nan,
       P_Yrev_base=(counts['Hsafe_Yrev']+counts['Hrev_Yrev'])/total if total else np.nan))
    elig.append(dict(mode=mode,eligible_climates=clim,
       n_climates=len(clim),min_supported_hour_fraction=float(p.supported_hour_fraction.min()),
       min_supported_cycle_fraction=float(p.supported_cycle_fraction.min())))
P=pd.DataFrame(pair_rows);S=pd.DataFrame(summary);CORR=pd.DataFrame(corr)
P.to_csv(OUT/'ImCOF_wholeH_wholeY_pairwise_transfer.csv',index=False)
S.to_csv(OUT/'ImCOF_wholeH_wholeY_transfer_summary.csv',index=False)
CORR.to_csv(OUT/'ImCOF_wholeH_wholeY_correspondence.csv',index=False)
print('Pairwise transfer:');display(S)
print('Within-representation H–Y correspondence (different statistic):');display(CORR)


## Predeclared support/coverage diagnostics (not primary exclusions)

Because 36 climates provide only a limited number of independent geographic units, pair-level counts are **not** treated as independent observations. The sensitivity below shows results after excluding climates with fewer than specified supported **hourly transitions** and supported **cycles**. Thresholds are diagnostics; they are **not** selected to maximize the effect. A source-data intervention cannot repair incomplete domain coverage.

In [ ]:
def transfer_counts(z):
    z=z[~(z.H_tie|z.Y_tie)].copy()
    n=len(z);safe=z[~z.H_reversal];reversedH=z[z.H_reversal]
    yrev=int(z.Y_reversal.sum());
    return dict(n_pairs=n,P_Yrev_base=yrev/n if n else np.nan,
      n_Hsafe=int(len(safe)),P_Yrev_given_Hsafe=float(safe.Y_reversal.mean()) if len(safe) else np.nan,
      n_Hreversed=int(len(reversedH)),P_Yrev_given_Hrev=float(reversedH.Y_reversal.mean()) if len(reversedH) else np.nan)
checks=[]
for (mode,pp) in P.groupby('operating_mode'):
    for mintr,mincy in [(0,0),(100,1),(1000,10),(5000,100)]:
        q=pp[(pp.min_hour_transitions>=mintr)&(pp.min_cycle_count>=mincy)]
        checks.append(dict(operating_mode=mode,min_hourly_transitions_per_climate=mintr,
          min_supported_cycles_per_climate=mincy,**transfer_counts(q)))
ROB=pd.DataFrame(checks)
ROB.to_csv(OUT/'ImCOF_wholeH_wholeY_coverage_diagnostic.csv',index=False)
display(ROB)


## Gates and provenance record

**Claims allowed:** representation-conditioned within-model H/Y pairwise comparisons on documented common climate support.  
**Claims not allowed:** measured ImCOF reactor validation; H as productivity; independent climate pairs; physical equivalence of H hourly states and Y cycle support; extrapolation to unsupported weather conditions.  
**Further independent process validation:** would require measured cycle-resolved ImCOF Y under matched weather/controller conditions.

In [ ]:
import json
record={
 'source_run':str(RUN), 'material':MATERIAL, 'input_files':{k:str(v) for k,v in PATHS.items()},
 'equilibrium_master':str(EQ_MASTER),'p0_bounds':str(P0_BOUNDS),
 'equilibrium_header':{'sheet':hit[0],'header_row_zero_based':int(hit[1])},
 'measured_T_knots_C':MEASURED_T_KNOTS.tolist(),
 'P0_gate':'PASS' if G1.dropna(subset=['abs_err']).abs_err.max()<1e-8 else 'FAIL',
 'P0_regression_rows':int(len(G1)),
 'primary_Y_gate':'PASS_SAVED_PRIMARY_SCHEMA_AND_CYCLE_COUNTS',
 'H_definition':'0.5 * total absolute difference across consecutive 1-hour weather states in common single/tensor q support',
 'process_alignment':'same full-range single/tensor q functions and identical climate names; H-hourly and Y-cycle state support differ',
 'n_climates_wholeH':int(WHOLE_H.climate.nunique()),
 'n_modes':int(S.operating_mode.nunique()),
 'limits':['H is not Y','not real process validation','no additive lower/upper H assumption','modeled Y itself is assumption dependent']
}
(OUT/'ImCOF_wholeH_Y_transfer_manifest.json').write_text(json.dumps(record,indent=2,ensure_ascii=False),encoding='utf-8')
print('SAVED:',OUT)
print('\n'.join(sorted(p.name for p in OUT.iterdir())))
